Authentification

In [ ]:
# token
import os

os.environ["DAGSHUB_TOKEN"] = "YOUR_DAGSHUB_TOKEN"

# Install MLFlow & the DagsHub python client

In [ ]:
%pip install -q dagshub mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.0/254.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 629.7/629.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

# Use the DagsHub client to setup connection information for MLflow

In [ ]:
pip install python-dotenv

In [ ]:
import dagshub
dagshub.auth.add_app_token(os.getenv("DAGSHUB_TOKEN"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import mlflow
import pandas as pd
import time
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Dagshub init
dagshub.init(repo_owner='your_repo_owner', repo_name='your_repo_name', mlflow=True)

# End of mlflow previous session
mlflow.end_run()

from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = mlflow.set_experiment("VGGNet_30")

print(experiment)

with mlflow.start_run():
    print("Training model start...")

    start_time = time.time()

    # Auto tracking activation
    mlflow.sklearn.autolog(log_models=True)

    from google.colab import drive
    drive.mount('/content/drive')

    # Data manipulation
    import librosa
    import numpy as np

    # Deep Learning
    import tensorflow as tf
    from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.utils import to_categorical
    from tensorflow.image import resize
    from tensorflow.keras.models import load_model
    from tensorflow.keras.losses import CategoricalCrossentropy

    # Data Visualization
    import librosa.display
    import matplotlib.pyplot as plt

    # Image Processing
    from skimage.transform import resize

    labels_file = "/path/to/train_labels.csv"  # Path to CSV file
    audio_dir = "/path/to/train_audios_wav"    # Path to audio files

    # Load labels
    labels_df = pd.read_csv(labels_file)

    # Get classes (diagnosis_control, diagnosis_mci, diagnosis_adrd)
    classes = ['diagnosis_control', 'diagnosis_mci', 'diagnosis_adrd']



    # Load and preprocess data
    def trim_silence(audio_data, sample_rate, top_db=50):
      edges = librosa.amplitude_to_db(np.abs(audio_data), ref=np.max)
      ref_value = np.max(edges[np.nonzero(edges)])
      y_trimmed, _ = librosa.effects.trim(audio_data, top_db=top_db, ref=ref_value)
      return y_trimmed

    def pad_audio(audio_data, sample_rate, target_duration):
      current_duration = librosa.get_duration(y=audio_data, sr=sample_rate)
      if current_duration < target_duration:
          padding_duration = target_duration - current_duration
          padding_samples = int(padding_duration * sample_rate)
          padded_audio = np.pad(audio_data, (0, padding_samples), 'constant')
          return padded_audio
      else:
          return audio_data

    def normalize_audio(audio_data):
      if len(audio_data) == 0:
          return audio_data
      max_amplitude = np.max(np.abs(audio_data))
      normalized_audio = audio_data / max_amplitude
      return normalized_audio

    def load_and_preprocess_data(audio_dir, labels_df, target_shape=(96, 1366), min_duration=30):
      data = []
      labels = []
      durations = []
      dropped_count = 0
      target_duration = 30

      for _, row in labels_df.iterrows():
          file_path = os.path.join(audio_dir, f"{row['uid']}.wav")
          if not os.path.exists(file_path):
              print(f"Missing audio file: {file_path}")
              continue
          audio_data, sample_rate = librosa.load(file_path, sr=None)
          audio_data = normalize_audio(audio_data)
          audio_data = pad_audio(audio_data, sample_rate, target_duration)
          duration = librosa.get_duration(y=audio_data, sr=sample_rate)
          durations.append(duration)
          if duration < min_duration:
              print(f"Skipping short audio: {file_path} ({duration:.2f} seconds)")
              dropped_count += 1
              continue
          mel_spectrogram = librosa.feature.melspectrogram(y=audio_data, sr=sample_rate)
          mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
          mel_spectrogram = np.expand_dims(mel_spectrogram, axis=-1)
          mel_spectrogram = resize(mel_spectrogram, target_shape)
          data.append(mel_spectrogram)
          labels.append([row['diagnosis_control'], row['diagnosis_mci'], row['diagnosis_adrd']])
      return np.array(data), np.array(labels), dropped_count, durations

    X, y, dropped_count, durations = load_and_preprocess_data(audio_dir, labels_df, min_duration=30)
    remaining_files = sum(1 for duration in durations if duration >= 30)

    # Train, Val Split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    input_shape = X_train[0].shape
    input_layer = Input(shape=input_shape)

    # Convolutional layers with adjusted pooling
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(input_layer)
    x = MaxPooling2D((2, 4))(x)  # Reduced pool size
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 4))(x)  # Reduced pool size
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 4))(x)  # Reduced pool size
    x = Conv2D(1024, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((3, 5))(x)  # Reduced pool size
    x = Conv2D(2048, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((4, 4))(x)  # Reduced pool size
    x = Conv2D(1024, (1, 1), activation='relu', padding='same')(x)

    # Fully connected layers
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    output_layer = Dense(len(classes), activation='softmax')(x)

    # Create model
    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
      optimizer=Adam(learning_rate=0.001),
      loss=CategoricalCrossentropy(),
      metrics=['accuracy']
    )

    history = model.fit(
      X_train, y_train,
      epochs=20,
      batch_size=32,
      validation_data=(X_val, y_val)
    )

    # Assuming 'history' contains training history (like from Keras)
    for epoch in range(len(history.history['accuracy'])):
       mlflow.log_metric("training_accuracy", history.history['accuracy'][epoch], step=epoch)
       mlflow.log_metric("validation_accuracy", history.history['val_accuracy'][epoch], step=epoch)
       mlflow.log_metric("training_loss", history.history['loss'][epoch], step=epoch)
       mlflow.log_metric("validation_loss", history.history['val_loss'][epoch], step=epoch)

    predictions = model.predict(X_val)

    # Model log
    signature = infer_signature(X_train, predictions)
    print("Input Signature: ", signature)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="VGGNet_30",
        registered_model_name="VGGNet_30",
        signature=signature,
    )

    print("...done !")
    print(f"Total training time : {time.time() - start_time:.2f}s")

    # Manuel log
    mlflow.log_param("Param name", "Value")

